# Session-Based & Real-Time Recommendations

Companion notebook for the [Session-Based & Real-Time Recommendations lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/04-session-based-and-realtime).

We implement a simplified SASRec-style self-attention session encoder, a contextual ε-greedy bandit, and simulate the real-time feature pipeline pattern. Pure NumPy / Python.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(42)

## 1 — Causal self-attention over item sequences

SASRec uses causal (left-to-right) self-attention so the model can only attend to items that appeared earlier in the session. We implement a single-head version.

In [ ]:
def causal_self_attention(Q, K, V):
    """Single-head causal self-attention. Q,K,V: (T, d)"""
    T, d = Q.shape
    scores = (Q @ K.T) / d**0.5                        # (T, T)
    mask = np.triu(np.full((T, T), -1e9), k=1)         # upper triangle = -inf
    scores += mask
    weights = np.exp(scores - scores.max(1, keepdims=True))
    weights /= weights.sum(1, keepdims=True)            # softmax
    return weights @ V                                  # (T, d)

# Toy session: 5 items, each represented by a d=8 embedding
T, d, n_items = 5, 8, 100
item_emb = rng.normal(size=(n_items, d))
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)

# Simulate a session: items [3, 17, 42, 8, 55]
session_ids = [3, 17, 42, 8, 55]
E = item_emb[session_ids]                              # (T, d) session embeddings

# Single-head attention with the session embeddings as Q=K=V (self-attention)
H = causal_self_attention(E, E, E)                     # (T, d)
print("Session embeddings shape:", E.shape)
print("Context-enriched embeddings shape:", H.shape)
print("Last position (used for next-item prediction):", H[-1].round(3))

## 2 — Next-item prediction

The last position's output from the attention layer is used to score all items in the catalog via dot product.

In [ ]:
def predict_next_items(session_ids, item_emb, top_k=5):
    E = item_emb[session_ids]
    H = causal_self_attention(E, E, E)
    query = H[-1]                                       # last position's output
    scores = item_emb @ query                           # dot product with all items
    # mask out items already in the session
    scores[session_ids] = -1e9
    return np.argsort(-scores)[:top_k]

top5 = predict_next_items(session_ids, item_emb)
print(f"Session: {session_ids}")
print(f"Top-5 predicted next items: {top5.tolist()}")

## 3 — ε-greedy exploration

Real-time recommenders inject random exploration to discover new preferences. ε-greedy: with probability ε, pick a random item; otherwise pick the top-ranked item.

In [ ]:
def epsilon_greedy_recommend(scores, epsilon=0.1, n_items_total=100, seed=None):
    rng_ = np.random.default_rng(seed)
    if rng_.random() < epsilon:
        # Explore: pick a random item (not already in top)
        chosen = rng_.integers(0, n_items_total)
        label = "EXPLORE"
    else:
        # Exploit: pick the highest-scoring item
        chosen = int(np.argmax(scores))
        label = "EXPLOIT"
    return chosen, label

# Simulate 1000 recommendation rounds, epsilon=0.15
all_scores = item_emb @ item_emb[session_ids[-1]]     # scores from last item's similarity
explore_count = sum(
    1 for _ in range(1000)
    if epsilon_greedy_recommend(all_scores, epsilon=0.15, seed=i)[1] == "EXPLORE"
    for i in [rng.integers(0,9999)]
)
print(f"Exploration rate over 1000 rounds: {explore_count/10:.1f}% (target ≈ 15%)")

## ✏️ Your turn

**Exercise.** Implement `masked_mean_pooling(session_ids, item_emb)` that computes the mean of the session item embeddings — the simplest possible session representation (no attention). Then compare its top-5 next-item predictions with the attention-based model above.

Does the mean pooling give different results? Why or why not?

In [ ]:
def masked_mean_pooling(session_ids, item_emb):
    # TODO(you): return the mean embedding of the items in session_ids
    return ...

def predict_topk_mean(session_ids, item_emb, top_k=5):
    query = masked_mean_pooling(session_ids, item_emb)
    scores = item_emb @ query
    scores[session_ids] = -1e9
    return np.argsort(-scores)[:top_k]

# Test your implementation
mean_top5 = predict_topk_mean(session_ids, item_emb)
print("Attention top-5:", top5.tolist())
print("Mean-pool top-5:", mean_top5.tolist())
print("Overlap:", len(set(top5.tolist()) & set(mean_top5.tolist())))

In [ ]:
# Assertion cell — passes silently when correct
query = masked_mean_pooling(session_ids, item_emb)
assert query.shape == (d,), f"Expected shape ({d},), got {query.shape}"
assert np.allclose(query, item_emb[session_ids].mean(0)), "Mean should be the average of session item embeddings"
print("✓ masked_mean_pooling is correct")

<details>
<summary>Solution</summary>

```python
def masked_mean_pooling(session_ids, item_emb):
    return item_emb[session_ids].mean(0)
```

Mean pooling is an unordered bag-of-items representation. It ignores the sequence order. Causal attention captures *transitions* (which item follows which), so it can model "user went from jazz → hip hop" as a direction, not just a set.
</details>